In [5]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import re
from nltk.stem import WordNetLemmatizer

In [14]:
df = pd.read_csv("data/plots.csv")

In [15]:
data_clean = df.description.apply(lambda x: re.sub(r'[^\w\s]', '', x.lower()))
data_clean

0       orin boyd un duro policía de una comisaría del...
1       al llegar a un pequeño pueblo donde ha heredad...
2       una mujer finge su muerte en un intento de esc...
3       durante un memorial en la ciudad natal de su p...
4       las pruebas nucleares francesas irradian a una...
                              ...                        
4962    el conde drácula aterroriza a la tripulación d...
4963    para salvar su relación una mujer se somete a ...
4964    paul kersey vuelve a trabajar como vigilante d...
4965    jane parker conoce a tarzán durante una expedi...
4966    los hijos del jefe de policía brody protegen a...
Name: description, Length: 4967, dtype: str

In [16]:
# Defino una funcion que recibe un texto y devuelve el mismo texto sin singnos,
def clean_text_round1(text):
    # pasa las mayusculas del texto a minusculas
    text = text.lower()
    # reemplaza texto entre corchetes por espacio en blanco.. ¿ y \% no se..
    text = re.sub(r'\[.*?¿\]%', ' ', text)
    # reemplaza signos de puntuacion por espacio en blanco.. %s -> \S+ es cualquier caracter que no sea un espacio en blanco
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # remueve palabras que contienen numeros.
    text = re.sub(r'\w*\d\w*', '', text)
    #eliminar tildes del español
    text = re.sub(r'[áàäâ]', 'a', text)
    text = re.sub(r'[éèëê]', 'e', text)
    text = re.sub(r'[íìïî]', 'i', text)
    text = re.sub(r'[óòöô]', 'o', text)
    text = re.sub(r'[úùüû]', 'u', text)
    return text

# Defino una funcion anonima que al pasarle un argumento devuelve el resultado de aplicarle la funcion anterior a este mismo argumento
round1 = lambda x: clean_text_round1(x)

# Dataframe que resulta de aplicarle a las columnas la funcion de limpieza
data_clean = pd.DataFrame(data_clean.apply(round1))

In [17]:
data_clean

,description
0,orin boyd un duro policia de una comisaria del...
1,al llegar a un pequeño pueblo donde ha heredad...
2,una mujer finge su muerte en un intento de esc...
3,durante un memorial en la ciudad natal de su p...
4,las pruebas nucleares francesas irradian a una...
...,...
4962,el conde dracula aterroriza a la tripulacion d...
4963,para salvar su relacion una mujer se somete a ...
4964,paul kersey vuelve a trabajar como vigilante d...
4965,jane parker conoce a tarzan durante una expedi...


In [18]:
# Hacemos una segunda vuelta de limpieza
def clean_text_round2(text):
    # Sacamos comillas, los puntos suspensivos, <<, >>
    text = re.sub('[‘’“”…«»]', '', text)
    text = re.sub('\n', ' ', text)
    return text

round2 = lambda x: clean_text_round2(x)

data_clean = pd.DataFrame(data_clean.description.apply(round2))

In [19]:
data_clean

,description
0,orin boyd un duro policia de una comisaria del...
1,al llegar a un pequeño pueblo donde ha heredad...
2,una mujer finge su muerte en un intento de esc...
3,durante un memorial en la ciudad natal de su p...
4,las pruebas nucleares francesas irradian a una...
...,...
4962,el conde dracula aterroriza a la tripulacion d...
4963,para salvar su relacion una mujer se somete a ...
4964,paul kersey vuelve a trabajar como vigilante d...
4965,jane parker conoce a tarzan durante una expedi...


In [24]:
# lematizador en español usando spacy
import spacy
nlp = spacy.load('es_core_news_sm')
def lemmatize_text(text):
    doc = nlp(text)
    lemmatized = ' '.join([token.lemma_ for token in doc])
    return lemmatized

# lematizamos el texto
round3 = lambda x: lemmatize_text(x)

# eliminamos stopwords
nltk.download('punkt_tab')
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))
def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

round4 = lambda x: remove_stopwords(x)

data_final =  pd.DataFrame(data_clean.description.apply(round3).apply(round4))

data_final

[nltk_data] Downloading package punkt_tab to /home/javier/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /home/javier/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,description
0,orin boyd duro policia comisaria centro ciudad...
1,llegar pequeño pueblo haber heredar mansion ru...
2,mujer finge muerte intento escapar horrible ma...
3,memorial ciudad natal padre nacido kentucky jo...
4,prueba nuclear francés irradiar iguana convert...
...,...
4962,conde dracular aterrorizar tripulacion nave es...
4963,salvar relacion mujer someter cirugia plastico
4964,paul kersey volver trabajar vigilante justicia...
4965,janir parker conocer tarzar expedicion africa ...


In [25]:
from gensim.models.phrases import Phrases, Phraser

input = [row.split() for row in data_final['description']] # separamos en una lista

phrases = Phrases(input, min_count=30, progress_per=1000)

bigram = Phraser(phrases)

sentences = bigram[input] # obtenemos las palabras junto con bigramas

In [27]:
%%time
import multiprocessing

from gensim.models import Word2Vec

cores = multiprocessing.cpu_count()

w2v_model = Word2Vec(min_count=20, # ignora palabras cuya frecuencia es menor a esta
                     window=2, # tamanio de la ventana de contexto
                     vector_size=300, # dimension del embedding
                     sample=6e-5, # umbral para downsamplear palabras muy frecuentes
                     alpha=0.03, # tasa de aprendizaje inicial (entrenamiento de la red neuronal)
                     min_alpha=0.0007, # tasa de aprendizaje minima
                     negative=20, # penalidad de palabras muy frecuentes o poco informaitvas
                     workers=cores) # numero de cores para entrenar el modelo

w2v_model.build_vocab(sentences, progress_per=10000) # construye el vocabulario

### ENTRENA EL MODELO
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

CPU times: user 2.42 s, sys: 59.1 ms, total: 2.48 s
Wall time: 1.45 s


(248511, 2112030)

In [30]:
w2v_model.wv.most_similar(positive=["mujer"])

[('joven', 0.9996968507766724),
 ('relacion', 0.9996939301490784),
 ('navidad', 0.9996867775917053),
 ('dia', 0.9996842741966248),
 ('mano', 0.9996824860572815),
 ('novio', 0.9996806979179382),
 ('novela', 0.99968022108078),
 ('mejor', 0.9996769428253174),
 ('mayor', 0.9996755719184875),
 ('aprender', 0.9996731877326965)]

In [31]:
w2v_model.wv.most_similar(positive=["hombre"])

[('tambien', 0.9996597766876221),
 ('mayor', 0.9996493458747864),
 ('desarrollar', 0.9996485114097595),
 ('alli', 0.9996482133865356),
 ('bruja', 0.9996455907821655),
 ('solo', 0.9996441006660461),
 ('competir', 0.999642550945282),
 ('violento', 0.999642014503479),
 ('pareja', 0.9996405839920044),
 ('romance', 0.999639630317688)]

## Segunda parte, simplificar la query del usuario usando TF-IDF

In [34]:
users_df = pd.read_csv("data/perfiles_usuarios.csv")

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Materializamos sentences como lista (es iterable pero por las dudas)
sentences_list = [list(s) for s in sentences]

# Pipeline idéntico al de las sinopsis, terminando con bigramas
def preprocesar_query(texto):
    t = clean_text_round1(texto)
    t = clean_text_round2(t)
    t = lemmatize_text(t)
    t = remove_stopwords(t)
    return list(bigram[t.split()])

# TF-IDF ajustado sobre las sinopsis ya preprocesadas (mismo espacio léxico que W2V)
sinopsis_str = [' '.join(s) for s in sentences_list]
vectorizer = TfidfVectorizer(min_df=2, max_df=0.8)
vectorizer.fit(sinopsis_str)
vocab = np.array(vectorizer.get_feature_names_out())

def top_terms(query, top_n=8):
    q_tokens = preprocesar_query(query)
    q_vec = vectorizer.transform([' '.join(q_tokens)]).toarray().flatten()
    idx = np.argsort(-q_vec)
    terms = []
    for i in idx:
        if q_vec[i] == 0:
            break
        w = vocab[i]
        if w in w2v_model.wv:        # filtramos a palabras conocidas por W2V
            terms.append((w, float(q_vec[i])))
        if len(terms) == top_n:
            break
    return terms

users_df['top_terms'] = users_df['query'].apply(top_terms)
users_df[['nombre', 'query', 'top_terms']].head(3)

,nombre,query,top_terms
0,Valentina,Quiero una película donde una mujer enfrenta u...,"[(alguien, 0.37083027009141273), (amenaza, 0.3..."
1,Rodrigo,Busco algo basado en hechos reales sobre corru...,"[(hecho, 0.43767679411962984), (corrupcion, 0...."
2,Camila,Una comedia donde la relación entre dos person...,"[(comedia, 0.40488381028974507), (forma, 0.384..."


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

# Vector de la query: promedio ponderado por TF-IDF de los embeddings de sus top terms
def vector_query(terms):
    if not terms:
        return None
    vecs  = np.array([w2v_model.wv[w] for w, _ in terms])
    pesos = np.array([p for _, p in terms])
    return np.average(vecs, axis=0, weights=pesos)

# Vector de cada película: centroide de los tokens conocidos por W2V
def vector_pelicula(tokens):
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else None

movie_vecs = [vector_pelicula(s) for s in sentences_list]
valid_idx  = [i for i, v in enumerate(movie_vecs) if v is not None]
movie_matrix = np.vstack([movie_vecs[i] for i in valid_idx])

# Recomendador: top 5 películas más cercanas, excluyendo el historial del usuario
hist_cols = [c for c in users_df.columns if c.startswith('pelicula_')]

def recomendar(row, top_n=5):
    qv = vector_query(row['top_terms'])
    if qv is None:
        return []
    historial = set(row[hist_cols].tolist())
    sims = cosine_similarity(qv.reshape(1, -1), movie_matrix).flatten()
    order = np.argsort(-sims)
    recs = []
    for j in order:
        i = valid_idx[j]
        name = df.iloc[i]['name']
        if name in historial:
            continue
        recs.append((name, float(sims[j])))
        if len(recs) == top_n:
            break
    return recs

users_df['recomendaciones'] = users_df.apply(recomendar, axis=1)

# Mostramos los 14 usuarios
for _, row in users_df.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']}) — \"{row['query']}\"")
    print(f"  top terms: {[t for t,_ in row['top_terms'][:5]]}")
    for name, score in row['recomendaciones']:
        print(f"   • {name}  ({score:.3f})")


Valentina (definido) — "Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano"
  top terms: ['alguien', 'amenaza', 'pelicula', 'querer', 'enfrentar']
   • El grito 2  (1.000)
   • ¡Oye Arnold! La película  (1.000)
   • BloodRayne  (1.000)
   • ¿Hacemos una porno?  (1.000)
   • Lilo &amp; Stitch  (1.000)

Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
  top terms: ['hecho', 'corrupcion', 'politico', 'basado', 'real']
   • Maleficio  (1.000)
   • El lobo de Wall Street  (1.000)
   • Silkwood  (1.000)
   • Protegidos por su enemigo  (1.000)
   • El asesinato de Richard Nixon  (1.000)

Camila (definido) — "Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental"
  top terms: ['comedia', 'forma', 'empezar', 'relacion', 'persona']
   • Caravana al Este  (1.000)
   • Pleasantville  (1.000)
   • La posesión  (1.000)
   • Operación Cóndor  (1.000)
   • Los compadres  (1.000)

